In [1]:
import re 
import ast
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from pre_processing_utils import Utils

In [3]:
folder_path = Path.cwd().parent / "data" / "new"

dict_worker_id = {}
for subdir in folder_path.iterdir():
    if subdir.is_dir():
        df = pd.read_csv(subdir / "Request.csv")
        worker_id_list = []
        for key, row in df.iterrows():
            value = row['unique_id']
            if isinstance(value, str):
                value_ = value.split(':')[0]
                if value_ not in worker_id_list:
                    worker_id_list.append(value_)
        dict_worker_id[subdir.name] = {
            idx  + 1: value for idx, value in enumerate(worker_id_list)
        }

#dict_worker_id

In [11]:
folder_path = Path.cwd().parent / "data/new" 

df = pd.DataFrame()
for subdir in folder_path.iterdir():
    if subdir.is_dir():
        if "new" not in subdir.name:
            print(f"Processing {subdir.name}...")
            df1 = pd.read_csv(subdir / "NestedGameTrial.csv")
            df1["session"] = subdir.name
            
            df = pd.concat([df, df1], ignore_index=True)

Processing 05-25-batch-1...


In [16]:
df.columns

Index(['_initial_assets', 'vars', 'id', 'creation_time', 'failed',
       'failed_reason', 'time_of_death', 'type', 'origin_id', 'network_id',
       'contents', 'complete', 'node_id', 'participant_id', 'module_state_id',
       'trial_maker_id', 'definition', 'finalized', 'is_repeat_trial', 'score',
       'performance_reward', 'parent_trial_id', 'answer', 'propagate_failure',
       'response_id', 'repeat_trial_index', 'n_repeat_trials', 'time_taken',
       'time_credit_before_trial', 'time_credit_after_trial',
       'time_credit_from_trial', 'progress_before_trial',
       'progress_after_trial', 'async_post_trial_required',
       'async_post_trial_requested', 'async_post_trial_complete',
       'async_post_trial_failed', 'block_position', 'block', 'class',
       'object_type', 'outer_game', 'inner_game', 'transition', 'wait',
       'outer_proposal', 'wait_1', 'wait_2', 'wait_3', 'wait_4',
       'inner_proposal', 'wait_5', 'wait_6', 'wait_7', 'wait_8', 'reward',
       'outer_

In [15]:
df.head()

,_initial_assets,vars,id,creation_time,failed,failed_reason,time_of_death,type,origin_id,network_id,...,inner_accept_answer,wait_9,summary,wait_10,wait_11,wait_12,wait_13,wait_14,wait_15,session
0,{},{},48,2026-05-26 00:20:21,True,"fail_on_timeout, UnsuccessfulEndPage, while_lo...",2026-05-26 00:25:35,NestedGameTrial,29,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,05-25-batch-1
1,{},{},49,2026-05-26 00:20:21,True,"fail_on_timeout, UnsuccessfulEndPage, while_lo...",2026-05-26 00:25:36,NestedGameTrial,29,14,...,Accept,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,05-25-batch-1
2,{},{},61,2026-05-26 00:21:16,True,"fail_on_timeout, UnsuccessfulEndPage, while_lo...",2026-05-26 00:25:35,NestedGameTrial,30,14,...,NaN,NaN,"{""outer_proposer"": 2, ""outer_responder"": 4, ""o...",NaN,NaN,NaN,NaN,NaN,NaN,05-25-batch-1
3,{},{},62,2026-05-26 00:21:16,True,"fail_on_timeout, UnsuccessfulEndPage, while_lo...",2026-05-26 00:25:35,NestedGameTrial,30,14,...,Accept,NaN,"{""outer_proposer"": 2, ""outer_responder"": 4, ""o...",NaN,NaN,NaN,NaN,NaN,NaN,05-25-batch-1
4,{},{},80,2026-05-26 00:22:04,True,"fail_on_timeout, UnsuccessfulEndPage, while_lo...",2026-05-26 00:25:35,NestedGameTrial,34,14,...,NaN,NaN,"{""outer_proposer"": 2, ""outer_responder"": 4, ""o...",NaN,NaN,NaN,NaN,NaN,NaN,05-25-batch-1


In [17]:
df['worker_id'] = df['participant_id'].map(dict_worker_id['05-25-batch-1'])

In [20]:
df.groupby('worker_id')['complete'].value_counts()

worker_id                 complete
60e282f9202a8126e2ea0315  True        25
66cff35bdee873a6ca915b84  False        1
670dc79e7b17f1b1ac7bdf95  False        1
671bbaf1bec15d847b871b9b  True         6
                          False        1
672c57a8ee24c76a011bf4d7  True         4
67b631294f6039a20633b917  True        25
67f2e613c09bbf379bd208ea  False        1
                          True         1
69808f7c8ff246c7ce92f77c  True         6
                          False        1
6993957ffe43ec54a046b703  False        1
                          True         1
699d0fdf43b9ecfcafe7e52a  True         5
6a036a8dcb5fdb1ea688ba0a  True         5
6a048950bab3f5d03e6ef938  True         4
Name: count, dtype: int64

In [30]:
finished_waiting = ['6a036a8dcb5fdb1ea688ba0a',
       '699d0fdf43b9ecfcafe7e52a', '68d4401a239d0533f8a66382',
       '5e8a1827052175000938eb91', '671bbaf1bec15d847b871b9b',
       '69808f7c8ff246c7ce92f77c', '670dc79e7b17f1b1ac7bdf95',
       '66cff35bdee873a6ca915b84', '6651e26558b5973c2cff426b',
       '69ee2e8675301578c95e1e60', '6a048950bab3f5d03e6ef938',
       '672c57a8ee24c76a011bf4d7', '6a0df5493e7bdfb3e8ad4caa',
       '66324d7b75233b6df004695d', '69ff52565942ff463af813fa',
       '69ef194605c1cba63afeba0b', '67b631294f6039a20633b917',
       '60e282f9202a8126e2ea0315', '65b571deec1559505fc20ac3',
       '5563984afdf99b672b5749b6', '6993957ffe43ec54a046b703',
       '67f2e613c09bbf379bd208ea', '63ffcb367b500c516d0648a2',
       '606e3cce02e2e8a8c965ca75']
kept_waiting = [x for x in finished_waiting if x not in proceeded]
print("Finished waiting but not proceeded:", len(kept_waiting))
kept_waiting

Finished waiting but not proceeded: 12


['68d4401a239d0533f8a66382',
 '5e8a1827052175000938eb91',
 '6651e26558b5973c2cff426b',
 '69ee2e8675301578c95e1e60',
 '6a0df5493e7bdfb3e8ad4caa',
 '66324d7b75233b6df004695d',
 '69ff52565942ff463af813fa',
 '69ef194605c1cba63afeba0b',
 '65b571deec1559505fc20ac3',
 '5563984afdf99b672b5749b6',
 '63ffcb367b500c516d0648a2',
 '606e3cce02e2e8a8c965ca75']

In [22]:
proceeded = df['worker_id'].unique()
proceeded

array(['699d0fdf43b9ecfcafe7e52a', '6a036a8dcb5fdb1ea688ba0a',
       '671bbaf1bec15d847b871b9b', '69808f7c8ff246c7ce92f77c',
       '670dc79e7b17f1b1ac7bdf95', '66cff35bdee873a6ca915b84',
       '672c57a8ee24c76a011bf4d7', '6a048950bab3f5d03e6ef938',
       '60e282f9202a8126e2ea0315', '67b631294f6039a20633b917',
       '67f2e613c09bbf379bd208ea', '6993957ffe43ec54a046b703'],
      dtype=object)

In [28]:
df[df['failed'] == False]['worker_id'].unique()

array(['60e282f9202a8126e2ea0315', '67b631294f6039a20633b917'],
      dtype=object)